[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/04_chafee_infante.ipynb)

# Chafee–Infante equation

The Chafee–Infante study learns latent models of dimensions 1, 2, and 3
for a 64-dimensional discretization. The release preserves the fine and
coarsened Morse representations, regions of attraction, and the 45-run
basin-classification study. All saved runs have two attracting minimal
sets.

The manuscript's bifurcation diagram is intentionally treated as a
static contextual figure: its source data and generator were not
preserved, and this companion repository does not claim to reproduce it.


In [ ]:
# Colab keeps a checkout so the artifact manifest remains available.
import os
import shutil
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/latent_dynamics")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "paper",
             "https://github.com/begelb/latent_dynamics.git", str(root)],
            check=True,
        )
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cmgdb==1.3.3+fork.3",
         "--find-links", "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.3"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    os.chdir(root)


## Load the released Chafee–Infante bundle


In [ ]:
import json
import pandas as pd
from latentdynamics.replay import load_experiment, show_image

chafee = load_experiment("chafee_infante_replay")
root = chafee.seed_dir.parent


## Fine and coarsened two-dimensional representations


In [ ]:
show_image(chafee.morse_dir / "morse_graph.png", width=540)
show_image(chafee.morse_dir / "morse_sets.png", width=660)
show_image(root / "coarsened" / "morse_roa_overlay.png", width=660)


## One- and three-dimensional latent models


In [ ]:
study = root / "latent_dimension_study"
for dim, figure in [("latent_1d", "morse_sets.png"),
                    ("latent_3d", "morse_sets_cubical_3d.png")]:
    mg = study / dim / "seed_0" / "MG_adaptive"
    show_image(mg / "morse_graph.png", width=520)
    show_image(mg / figure, width=660)


## Basin-classification replay


In [ ]:
stats = json.loads((root / "continuation_10000" / "updated_paper_statistics.json").read_text())
rows = []
for dim, record in stats["summary_by_dimension"].items():
    summary = record["completed_combined_correct"]
    rows.append({"latent dimension": int(dim), "runs": summary["n"],
                 "correct mean (%)": summary["mean"],
                 "sample SD": summary["sample_standard_deviation"]})
pd.DataFrame(rows).sort_values("latent dimension").style.format(precision=2)


The displayed d=2 replay uses the archived reference computation; the
production parameter record is documented separately in
`docs/figure_contracts/chafee_infante.md`. Fresh retraining is
stochastic, and the saved d=2 fine result survives only as author figure
files rather than raw DOT/CSV.
